<a href="https://colab.research.google.com/github/amimulhasan/ML_project/blob/main/gradCam_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# A–Z COMPLETE GRAD-CAM CODE
# ================================

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from PIL import Image

# ------------------------
# SETTINGS
# ------------------------
dataset_dir = "/kaggle/input/brain-tumor-mri-data/brain-tumor-mri-dataset"
img_size = (128, 128)
expected_classes = 4
epochs = 10
batch_size = 64

# ------------------------
# LOAD DATA
# ------------------------
X, y = [], []
class_names = sorted(
    [d for d in os.listdir(dataset_dir)
     if os.path.isdir(os.path.join(dataset_dir, d))]
)

print("Classes found:", class_names)

if len(class_names) != expected_classes:
    raise ValueError(
        f"Expected {expected_classes} classes, found {len(class_names)}"
    )

for idx, class_name in enumerate(class_names):
    class_path = os.path.join(dataset_dir, class_name)
    for root, _, files in os.walk(class_path):
        for file in files:
            try:
                img = Image.open(os.path.join(root, file)).convert("RGB")
                img = img.resize(img_size)
                X.append(np.array(img, dtype=np.float32) / 255.0)
                y.append(idx)
            except:
                continue

X = np.array(X)
y = np.array(y)

print("Dataset loaded:", X.shape, y.shape)

# ------------------------
# TRAIN / VAL SPLIT
# ------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ------------------------
# MODEL PARAMETERS
# ------------------------
input_shape = (128, 128, 3)
patch_size = 16
num_patches = (128 // patch_size) ** 2
projection_dim = 64
transformer_layers = 4
num_heads = 8
num_classes = expected_classes

# ------------------------
# PATCH + ENCODER
# ------------------------
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patches = tf.reshape(patches, [batch, -1, patches.shape[-1]])
        return patches


class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

# ------------------------
# HYBRID MODEL
# ------------------------
def build_hybrid_model():
    inputs = layers.Input(shape=input_shape)

    # CNN BRANCH
    x_cnn = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    x_cnn = layers.MaxPooling2D()(x_cnn)
    x_cnn = layers.Conv2D(64, 3, activation="relu", padding="same")(x_cnn)
    x_cnn = layers.MaxPooling2D()(x_cnn)
    x_cnn = layers.Conv2D(128, 3, activation="relu", padding="same")(x_cnn)
    x_cnn = layers.MaxPooling2D()(x_cnn)
    x_cnn = layers.Flatten()(x_cnn)

    # ViT + GRU BRANCH
    patches = Patches(patch_size)(inputs)
    encoded = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded)
        attn = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim
        )(x1, x1)
        x2 = layers.Add()([attn, encoded])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        encoded = layers.Add()([x3, x2])

    x_vit = layers.LayerNormalization(epsilon=1e-6)(encoded)
    x_vit = layers.Flatten()(x_vit)
    x_vit = layers.Reshape((-1, x_vit.shape[-1]))(x_vit)
    x_vit = layers.GRU(256)(x_vit)

    # FUSION
    x = layers.Concatenate()([x_cnn, x_vit])
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.003)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)

model = build_hybrid_model()
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ------------------------
# TRAIN
# ------------------------
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size
)

# ------------------------
# GRAD-CAM FUNCTIONS
# ------------------------
# def find_last_conv_layer(model):
#     for layer in reversed(model.layers):
#         if isinstance(layer, layers.Conv2D):
#             return layer.name
#     return None


# def make_gradcam_heatmap(model, image, layer_name, class_index=None):
#     img_tensor = tf.expand_dims(image, 0)

#     grad_model = tf.keras.models.Model(
#         [model.inputs],
#         [model.get_layer(layer_name).output, model.output]
#     )

#     with tf.GradientTape() as tape:
#         conv_out, preds = grad_model(img_tensor)
#         if class_index is None:
#             class_index = tf.argmax(preds[0])
#         loss = preds[:, class_index]

#     grads = tape.gradient(loss, conv_out)
#     pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

#     conv_out = conv_out[0]
#     heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
#     heatmap = tf.maximum(heatmap, 0)
#     heatmap /= tf.reduce_max(heatmap) + 1e-9

#     return heatmap.numpy()


# def overlay_heatmap(image, heatmap, alpha=0.45):
#     heatmap = np.uint8(255 * heatmap)
#     jet = plt.cm.jet(heatmap / 255.0)[:, :, :3]
#     jet = tf.image.resize(jet, image.shape[:2]).numpy()
#     return np.clip(image * (1 - alpha) + jet * alpha, 0, 1)

# # ------------------------
# # HORIZONTAL GRAD-CAM PLOT (8 CLASS)
# # ------------------------
# last_conv = find_last_conv_layer(model)

# fig, axs = plt.subplots(
#     3, expected_classes,
#     figsize=(expected_classes * 4, 12)
# )

# fig.suptitle(
#     "Grad-CAM Visualization (Horizontal Class-wise)",
#     fontsize=18,
#     weight="bold"
# )

# for c in range(expected_classes):
#     idxs = np.where(y_val == c)[0]
#     idx = np.random.choice(idxs)

#     img = X_val[idx]
#     pred = np.argmax(model.predict(img[None], verbose=0))

#     heatmap = make_gradcam_heatmap(
#         model, img, last_conv, pred
#     )
#     overlay = overlay_heatmap(img, heatmap)

#     axs[0, c].imshow(img)
#     axs[0, c].set_title(f"Class {c}\nOriginal")
#     axs[0, c].axis("off")

#     axs[1, c].imshow(heatmap, cmap="jet")
#     axs[1, c].set_title("Grad-CAM")
#     axs[1, c].axis("off")

#     axs[2, c].imshow(overlay)
#     axs[2, c].set_title("Overlay")
#     axs[2, c].axis("off")

# axs[0, 0].set_ylabel("Original", fontsize=20, weight="bold")
# axs[1, 0].set_ylabel("Grad-CAM", fontsize=20, weight="bold")
# axs[2, 0].set_ylabel("Overlay", fontsize=20, weight="bold")

# plt.tight_layout(rect=[0, 0, 1, 0.94])
# plt.savefig("gradcam_horizontal_8_classes.png", dpi=1200)
# plt.show()

# print("✅ DONE: gradcam_horizontal_8_classes.png saved")
# ================================
# A–Z FINAL GRAD-CAM VISUALIZATION
# ================================

# ------------------------
# FIND LAST CONV LAYER
# ------------------------
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found.")

# ------------------------
# GRAD-CAM HEATMAP
# ------------------------
def make_gradcam_heatmap(model, image, layer_name, class_index=None):

    img_tensor = tf.expand_dims(image, 0)

    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tensor)
        if class_index is None:
            class_index = tf.argmax(preds[0])
        loss = preds[:, class_index]

    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy()

# ------------------------
# OVERLAY FUNCTION
# ------------------------
def overlay_heatmap(image, heatmap, alpha=0.45):

    heatmap = np.uint8(255 * heatmap)
    heatmap = plt.cm.jet(heatmap / 255.0)[:, :, :3]
    heatmap = tf.image.resize(heatmap, image.shape[:2]).numpy()

    overlay = image * (1 - alpha) + heatmap * alpha
    return np.clip(overlay, 0, 1)

# ------------------------
# HORIZONTAL GRAD-CAM (8 CLASS)
# ------------------------
last_conv = find_last_conv_layer(model)

fig, axs = plt.subplots(
    nrows=3,
    ncols=expected_classes,
    figsize=(expected_classes * 3.0, 8.5)   # 👈 narrower width
)

fig.suptitle(
    "Grad-CAM Visualization (Class-wise)",
    fontsize=22,
    fontweight="bold",
    y=0.98
)

for c in range(expected_classes):
    idxs = np.where(y_val == c)[0]
    idx = np.random.choice(idxs)

    img = X_val[idx]
    pred = np.argmax(model.predict(img[None], verbose=0))

    heatmap = make_gradcam_heatmap(
        model, img, last_conv, pred
    )
    overlay = overlay_heatmap(img, heatmap)

    # Original
    axs[0, c].imshow(img)
    axs[0, c].set_title(
        f"Class {c}",
        fontsize=16,
        fontweight="bold",
        pad=6
    )
    axs[0, c].axis("off")

    # Grad-CAM
    axs[1, c].imshow(heatmap, cmap="jet")
    axs[1, c].set_title(
        "Grad-CAM",
        fontsize=15,
        fontweight="bold",
        pad=6
    )
    axs[1, c].axis("off")

    # Overlay
    axs[2, c].imshow(overlay)
    axs[2, c].set_title(
        "Overlay",
        fontsize=15,
        fontweight="bold",
        pad=6
    )
    axs[2, c].axis("off")

# ------------------------
# ROW LABELS (BIG FONT)
# ------------------------
axs[0, 0].set_ylabel("Original", fontsize=18, fontweight="bold", labelpad=12)
axs[1, 0].set_ylabel("Grad-CAM", fontsize=18, fontweight="bold", labelpad=12)
axs[2, 0].set_ylabel("Overlay", fontsize=18, fontweight="bold", labelpad=12)

# ------------------------
# VERY TIGHT SPACING (KEY FIX)
# ------------------------
plt.subplots_adjust(
    wspace=0.015,   # 👈 columns extremely close
    hspace=0.10
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("gradcam_horizontal_8_classes3333.png", dpi=800)
plt.show()

print("✅ DONE: gradcam_horizontal_8_classes.png saved")
